# Train Smart Helmet Detection (YOLO11) — Colab + GPU ในเครื่อง (RTX 5060)

Dataset: [Helmet Detection_YOLOv8 v3](https://universe.roboflow.com/learning-evidence/helmet-detection_yolov8) (CC BY 4.0) — 2 คลาส: `With Helmet`, `Without Helmet`

### ตั้งค่าครั้งแรก (ทำใน terminal ของเครื่อง)
```
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128
pip install ultralytics roboflow notebook
```

### ทุกครั้งที่จะใช้
1. เปิด terminal แล้วรัน (เปิดทิ้งไว้ตลอดตอน train):
```
jupyter notebook --ServerApp.allow_origin=https://colab.research.google.com --port=8888 --ServerApp.port_retries=0
```
2. ก๊อป URL ที่ขึ้นมา (`http://localhost:8888/?token=...`)
3. ใน Colab: ลูกศร ▾ ข้างปุ่ม **Connect** มุมขวาบน → **Connect to a local runtime** → วาง URL → Connect
4. รันทีละ cell จากบนลงล่าง

## 1. เช็ค GPU
ต้องขึ้น `NVIDIA GeForce RTX 5060` ถ้าขึ้น error แปลว่ายังไม่ได้เชื่อม local runtime หรือ PyTorch ไม่ใช่แบบ cu128

In [ ]:
import torch
assert torch.cuda.is_available(), 'ไม่เจอ GPU: ติดตั้ง torch จาก https://download.pytorch.org/whl/cu128 และเช็คว่าเชื่อม local runtime แล้ว'
print(torch.cuda.get_device_name(0), '| torch', torch.__version__, '| CUDA', torch.version.cuda)

## 2. โฟลเดอร์เก็บงาน
dataset, ผล train และ `best.pt` จะอยู่ที่ `C:\Users\<ชื่อคุณ>\helmet-detection`

In [ ]:
from pathlib import Path
WORK = Path.home() / 'helmet-detection'
WORK.mkdir(exist_ok=True)
DATA_DIR = WORK / 'datasets' / 'helmet-detection_yolov8-3'
RUNS_DIR = WORK / 'runs'
print(WORK)

## 3. ดาวน์โหลด dataset (ครั้งแรกครั้งเดียว)
ต้องใช้ Roboflow API key (ฟรี): roboflow.com → Settings → API Keys → คัดลอก **Private API Key** แล้ววางในช่องที่ขึ้นมา

In [ ]:
from getpass import getpass
from roboflow import Roboflow

if not (DATA_DIR / 'data.yaml').exists():
    rf = Roboflow(api_key=getpass('Roboflow API key: '))
    rf.workspace('learning-evidence').project('helmet-detection_yolov8').version(3).download('yolov11', location=str(DATA_DIR))
DATA_YAML = str(DATA_DIR / 'data.yaml')
print(open(DATA_YAML).read())

## 4. Train
`yolo11n` ใช้เวลาประมาณ 15–25 นาทีบน RTX 5060 · ถ้าเจอ `CUDA out of memory` ให้ลด `batch` เป็น 8 · ถ้าค้างตอนเริ่ม ให้ตั้ง `workers=0`

In [ ]:
from ultralytics import YOLO

NAME = 'yolo11n'
model = YOLO(f'{NAME}.pt')
model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,      # หยุดก่อนถ้าไม่ดีขึ้น 15 epoch ติด
    device=0,         # RTX 5060
    workers=4,
    project=str(RUNS_DIR),
    name=NAME,
    exist_ok=True,
)

## 5. ประเมินผลบน test set
ตัวเลขสำหรับใส่รายงาน

In [ ]:
best = YOLO(RUNS_DIR / NAME / 'weights' / 'best.pt')
m = best.val(data=DATA_YAML, split='test', project=str(RUNS_DIR), name=f'{NAME}-test', exist_ok=True)
print(f'mAP50     : {m.box.map50:.3f}')
print(f'mAP50-95  : {m.box.map:.3f}')
print(f'Precision : {m.box.mp:.3f}')
print(f'Recall    : {m.box.mr:.3f}')
for i, c in enumerate(m.box.ap_class_index):
    print(f'{best.names[int(c)]:15s} AP50 = {m.box.ap50[i]:.3f}')

## 6. กราฟสำหรับรายงาน

In [ ]:
from IPython.display import Image, display
run = RUNS_DIR / NAME
for f in ['results.png', 'confusion_matrix.png', 'BoxF1_curve.png', 'F1_curve.png', 'val_batch0_pred.jpg']:
    if (run / f).exists():
        print(f); display(Image(str(run / f), width=800))

## 7. ลองทำนายรูปจาก test set

In [ ]:
imgs = sorted((DATA_DIR / 'test' / 'images').glob('*'))[:8]
best.predict([str(p) for p in imgs], conf=0.5, save=True, project=str(RUNS_DIR), name='predict', exist_ok=True)
for p in sorted((RUNS_DIR / 'predict').glob('*.jpg'))[:4]:
    display(Image(str(p), width=600))

## 8. เก็บ `best.pt`
ไฟล์โมเดลสำหรับใช้ต่อใน Back-end

In [ ]:
import shutil
shutil.copy(RUNS_DIR / NAME / 'weights' / 'best.pt', WORK / 'best.pt')
print('โมเดล:', WORK / 'best.pt')
print('กราฟทั้งหมด:', RUNS_DIR / NAME)